# EWC Space Delta-V · Issue 027 · Working demo

Link budgets, the rocket equation and cold-atom equivalent temperature: every number quoted in the edition.

**Code status: working demo.** Python 3, standard library only; nothing to install. Choose *Runtime → Run all*. The notebook prints every number quoted in the edition, then runs an 18-check self-test. Company figures are marked `PUBLISHED` in the code; everything else is marked `ASSUMED`.

**Precaution.** This is an educational model with stated assumptions, not a mission-grade tool and not any company's solver. Test it in your own environment and have qualified experts review it before any implementation. It is provided without warranty of fitness for any real mission, safety-critical or operational use.

**How the notebook is organised:** imports, constants, one block of functions for each part of the edition (A to E), the report that prints the numbers, the self-test, and finally the cell that runs them.

## 1. Imports
Only the standard library's `math` module is needed.

In [ ]:
import math

## 2. Constants and helpers
Physical constants and two decibel helpers used by every section. `K_B` is the Boltzmann constant, `G0` standard gravity, and the `R_*` and `MU_EARTH` values describe Earth, the geostationary orbit and the Moon.

In [ ]:
# ---------------------------------------------------------------- constants
C = 299_792_458.0            # speed of light, m/s
K_B = 1.380649e-23           # Boltzmann constant, J/K
G0 = 9.80665                 # standard gravity, m/s^2
MU_EARTH = 398_600.4418      # km^3/s^2
R_EARTH = 6371.0             # mean Earth radius, km
R_GEO = 42_164.17            # GEO radius, km
R_MOON = 384_400.0           # mean Earth-Moon distance, km
M_RB87 = 86.909180527 * 1.66053906660e-27   # kg


def db(x):
    return 10.0 * math.log10(x)


def undb(x_db):
    return 10.0 ** (x_db / 10.0)

## 3. A. Optical LEO downlink
Beam radius at the receiver, the fraction of a Gaussian beam the aperture collects, the pointing loss, and the power-optimal beam divergence (θ = √2 σ). `OPT` holds the published ATLAS-2 and OGS-2.5 figures and the assumed ones, and `optical_budget` combines them into a link margin at a given elevation.

In [ ]:
# ---------------------------------------------------------------- A. optical LEO downlink
def slant_range_km(alt_km, el_deg):
    """Slant range from a ground site to a satellite on a circular orbit."""
    e = math.radians(el_deg)
    return math.sqrt((R_EARTH + alt_km) ** 2 - (R_EARTH * math.cos(e)) ** 2) - R_EARTH * math.sin(e)


def gaussian_capture(w_m, a_m):
    """Fraction of a Gaussian beam (1/e^2 radius w) collected by a circular aperture of radius a."""
    return 1.0 - math.exp(-2.0 * a_m ** 2 / w_m ** 2)


def pointing_loss(sigma_rad, theta_rad):
    """Gaussian on-axis loss for a fixed pointing offset sigma and 1/e^2 half-divergence theta."""
    return math.exp(-2.0 * (sigma_rad / theta_rad) ** 2)


def optimal_half_divergence(sigma_rad):
    """Half-divergence that maximises received power for a fixed offset: theta = sqrt(2) * sigma."""
    return math.sqrt(2.0) * sigma_rad


OPT = dict(
    wavelength_m=1553.3e-9,      # PUBLISHED (ATLAS-2 page, Tx/Rx 1553.3 / 1536.6 nm)
    tx_power_w=2.0,              # PUBLISHED as "< 2 W"; the upper bound is used
    half_div_rad=63e-6,          # PUBLISHED: +/-63 urad at 1/e^2 (ATLAS-2)
    fine_pointing_rad=15e-6,     # PUBLISHED as "< 15 urad"; the bound is used
    rx_aperture_m=0.355,         # PUBLISHED: OGS-2.5 option A, 355 mm
    tx_optics_eff=0.7,           # ASSUMED
    rx_optics_eff=0.7,           # ASSUMED
    zenith_tau=0.10,             # ASSUMED clear-sky optical depth at 1550 nm
    rx_sensitivity_dbm=-36.0,    # ASSUMED, direct-detection OOK at 2.5 Gbps
    fade_margin_db=3.0,          # ASSUMED scintillation and implementation allowance
    altitude_km=500.0,           # ASSUMED LEO altitude
)


def optical_budget(el_deg, p=OPT):
    R_km = slant_range_km(p["altitude_km"], el_deg)
    R_m = R_km * 1e3
    w = R_m * p["half_div_rad"]                       # spot radius at the receiver
    a = p["rx_aperture_m"] / 2.0
    capture = gaussian_capture(w, a)
    point = pointing_loss(p["fine_pointing_rad"], p["half_div_rad"])
    atm = math.exp(-p["zenith_tau"] / math.sin(math.radians(el_deg)))
    tx_dbm = db(p["tx_power_w"] * 1e3)
    rx_dbm = (tx_dbm + db(capture) + db(point) + db(atm)
              + db(p["tx_optics_eff"]) + db(p["rx_optics_eff"]))
    margin = rx_dbm - p["rx_sensitivity_dbm"] - p["fade_margin_db"]
    return dict(el=el_deg, R_km=R_km, spot_radius_m=w, geo_loss_db=db(capture),
                point_loss_db=db(point), atm_loss_db=db(atm), rx_dbm=rx_dbm, margin_db=margin)

## 4. B. Deep-space RF link
Free-space path loss, parabolic-dish gain and the maximum bit rate of an S-band link with an omnidirectional spacecraft antenna. All spacecraft figures in `RF` are assumed.

In [ ]:
# ---------------------------------------------------------------- B. deep-space RF link
def fspl_db(d_m, f_hz):
    """Free-space path loss, Friis: (4 pi d f / c)^2 in dB."""
    return 20.0 * math.log10(4.0 * math.pi * d_m * f_hz / C)


def dish_gain_dbi(diam_m, f_hz, eff):
    """Parabolic dish gain: eff * (pi D f / c)^2 in dBi."""
    return db(eff * (math.pi * diam_m * f_hz / C) ** 2)


RF = dict(
    freq_hz=2.2e9,               # ASSUMED S-band downlink (company states S-band only)
    sc_tx_power_w=10.0,          # ASSUMED spacecraft transmitter power
    sc_antenna_dbi=0.0,          # ASSUMED omnidirectional (company: omnidirectional antennas)
    misc_loss_db=2.0,            # ASSUMED polarisation, pointing, atmosphere
    dish_eff=0.55,               # ASSUMED
    t_sys_k=100.0,               # ASSUMED system noise temperature at low elevation
    ebn0_req_db=2.5,             # ASSUMED coded-modulation threshold
    ebn0_margin_db=3.0,          # ASSUMED link margin
)


def rf_max_rate_bps(range_km, dish_m, p=RF):
    eirp_dbw = db(p["sc_tx_power_w"]) + p["sc_antenna_dbi"]
    loss = fspl_db(range_km * 1e3, p["freq_hz"])
    g_r = dish_gain_dbi(dish_m, p["freq_hz"], p["dish_eff"])
    pr_dbw = eirp_dbw - loss + g_r - p["misc_loss_db"]
    n0_dbw_hz = db(K_B * p["t_sys_k"])
    cn0 = pr_dbw - n0_dbw_hz
    rate_db = cn0 - (p["ebn0_req_db"] + p["ebn0_margin_db"])
    return dict(range_km=range_km, dish_m=dish_m, fspl_db=loss, gain_dbi=g_r,
                pr_dbw=pr_dbw, cn0_dbhz=cn0, rate_bps=10.0 ** (rate_db / 10.0))

## 5. C. Rocket equation
Circular-orbit speed, the two-burn Hohmann transfer (with an optional plane change at apogee), the burn to the lunar distance, the mass ratio from the rocket equation and the payload fraction for a stage with structural coefficient ε.

In [ ]:
# ---------------------------------------------------------------- C. rocket equation
def circ_speed(r_km):
    return math.sqrt(MU_EARTH / r_km)


def hohmann_dv(r1_km, r2_km, incl_deg=0.0):
    """Two-burn transfer between circular orbits; optional plane change combined at apogee."""
    a = 0.5 * (r1_km + r2_km)
    v_peri = math.sqrt(MU_EARTH * (2.0 / r1_km - 1.0 / a))
    v_apo = math.sqrt(MU_EARTH * (2.0 / r2_km - 1.0 / a))
    dv1 = v_peri - circ_speed(r1_km)
    v2 = circ_speed(r2_km)
    i = math.radians(incl_deg)
    dv2 = math.sqrt(v_apo ** 2 + v2 ** 2 - 2.0 * v_apo * v2 * math.cos(i))
    return dv1, dv2


def tli_dv(r1_km):
    """Burn from a circular orbit to an ellipse reaching the lunar distance."""
    a = 0.5 * (r1_km + R_MOON)
    return math.sqrt(MU_EARTH * (2.0 / r1_km - 1.0 / a)) - circ_speed(r1_km)


def mass_ratio(dv_kms, isp_s):
    return math.exp(dv_kms * 1e3 / (isp_s * G0))


def payload_fraction(dv_kms, isp_s, eps):
    """payload / initial mass for a single stage with structural coefficient eps."""
    mr = mass_ratio(dv_kms, isp_s)
    return (1.0 / mr - eps) / (1.0 - eps)

## 6. D. Cold-atom equivalent temperature
Converts between an equivalent temperature and the one-dimensional velocity spread of a rubidium-87 cloud.

In [ ]:
# ---------------------------------------------------------------- D. cold-atom equivalent temperature
def sigma_v(temp_k, mass_kg=M_RB87):
    """One-dimensional rms velocity spread for an equivalent temperature."""
    return math.sqrt(K_B * temp_k / mass_kg)


def equivalent_temp(sigma_v_ms, mass_kg=M_RB87):
    return mass_kg * sigma_v_ms ** 2 / K_B

## 7. E. Rarefied gas, free fall and fleet screening
Hard-sphere mean free path, Knudsen number and flow regime, free fall from rest, and the number of satellite pairs a conjunction screen must consider.

In [ ]:
# ---------------------------------------------------------------- E. rarefied gas, free fall, fleet screening
def mean_free_path_m(p_pa, t_k=300.0, d_m=3.7e-10):
    """Hard-sphere mean free path, lambda = k_B T / (sqrt(2) pi d^2 p). d is ASSUMED (nitrogen-like)."""
    return K_B * t_k / (math.sqrt(2.0) * math.pi * d_m ** 2 * p_pa)


def knudsen(p_pa, length_m, t_k=300.0, d_m=3.7e-10):
    return mean_free_path_m(p_pa, t_k, d_m) / length_m


def regime(kn):
    if kn < 0.01:
        return "continuum"
    if kn < 0.1:
        return "slip"
    if kn < 10.0:
        return "transitional"
    return "free-molecular"


def free_fall_m(t_s):
    return 0.5 * G0 * t_s ** 2


def screening_pairs(n):
    """Satellite pairs a conjunction screen must consider for a fleet of n."""
    return n * (n - 1) // 2

## 8. The report
Prints every number quoted in the article, section by section.

In [ ]:
# ---------------------------------------------------------------- report
def report():
    print("== A. Optical LEO downlink (reference design built from PUBLISHED ATLAS-2 / OGS-2.5 figures) ==")
    print("   NOT the ERMIS-3 flight terminal: ATLAS-1 beam parameters are not published.")
    lam = OPT["wavelength_m"]
    w0 = 0.04 / 2.0
    th_diff = lam / (math.pi * w0)
    print(f"   Diffraction-limited half-divergence for a 40 mm aperture (w0 = 20 mm): {th_diff*1e6:.1f} urad")
    print(f"   Published half-divergence {OPT['half_div_rad']*1e6:.0f} urad; fine pointing {OPT['fine_pointing_rad']*1e6:.0f} urad")
    opt_th = optimal_half_divergence(OPT["fine_pointing_rad"])
    pw = lambda th: (1.0 / th ** 2) * pointing_loss(OPT["fine_pointing_rad"], th)
    print(f"   Power-optimal half-divergence for a fixed {OPT['fine_pointing_rad']*1e6:.0f} urad offset: {opt_th*1e6:.1f} urad")
    print(f"   Power given up by using {OPT['half_div_rad']*1e6:.0f} urad instead: {db(pw(opt_th)/pw(OPT['half_div_rad'])):.1f} dB")
    print("   el(deg)  range(km)  spot radius(m)  geo(dB)  point(dB)  atm(dB)  Prx(dBm)  margin(dB)")
    for el in (10, 15, 30, 60, 90):
        b = optical_budget(el)
        print(f"   {el:5d}  {b['R_km']:9.0f}  {b['spot_radius_m']:13.1f}  {b['geo_loss_db']:7.1f}"
              f"  {b['point_loss_db']:8.2f}  {b['atm_loss_db']:7.2f}  {b['rx_dbm']:8.1f}  {b['margin_db']:9.1f}")

    print("\n== B. Deep-space RF link (S-band, omnidirectional spacecraft antenna; all spacecraft figures ASSUMED) ==")
    f = RF["freq_hz"]
    print(f"   Free-space loss at 20 million km, {f/1e9:.1f} GHz: {fspl_db(20e6*1e3, f):.1f} dB")
    for d in (15.0, 35.0):
        print(f"   Dish gain, {d:.0f} m, eff {RF['dish_eff']:.2f}: {dish_gain_dbi(d, f, RF['dish_eff']):.1f} dBi")
    ratio = (35.0 / 15.0) ** 2
    print(f"   Collecting-area ratio 35 m / 15 m: {ratio:.2f}  ({db(ratio):.2f} dB)")
    print("   range(km)       dish   C/N0(dBHz)   max rate (bit/s)")
    for rng in (R_MOON, 1e6, 5e6, 20e6):
        for d in (15.0, 35.0):
            r = rf_max_rate_bps(rng, d)
            print(f"   {rng:11,.0f}   {d:4.0f} m   {r['cn0_dbhz']:9.1f}   {r['rate_bps']:14.1f}")

    print("\n== C. Rocket equation (Isp 460 s is Blue Origin's stated BE-7 figure; 360 s is an illustrative comparison) ==")
    r_leo = R_EARTH + 185.0
    d1, d2 = hohmann_dv(r_leo, R_GEO, 0.0)
    d1i, d2i = hohmann_dv(r_leo, R_GEO, 28.5)
    dv_geo_planar = d1 + d2
    dv_geo_28 = d1i + d2i
    dv_tli = tli_dv(r_leo)
    print(f"   185 km circular speed: {circ_speed(r_leo):.3f} km/s; GEO circular speed: {circ_speed(R_GEO):.3f} km/s")
    print(f"   Hohmann LEO(185 km) -> GEO, planar: {d1:.3f} + {d2:.3f} = {dv_geo_planar:.3f} km/s")
    print(f"   Same with 28.5 deg plane change combined at apogee: {d1i:.3f} + {d2i:.3f} = {dv_geo_28:.3f} km/s")
    print(f"   Burn from 185 km circular to lunar distance: {dv_tli:.3f} km/s")
    for name, dv in (("GEO planar", dv_geo_planar), ("GEO with 28.5 deg", dv_geo_28), ("TLI", dv_tli)):
        for isp in (460.0, 360.0):
            mr = mass_ratio(dv, isp)
            pf = payload_fraction(dv, isp, 0.12)
            print(f"   {name:18s} Isp {isp:3.0f} s: v_e = {isp*G0/1e3:.2f} km/s, mass ratio {mr:.3f}, "
                  f"propellant fraction {1-1/mr:.3f}, payload/initial mass (eps 0.12) {pf:.3f}")
    print("   (payload/initial mass: structural coefficient eps = 0.12 ASSUMED, equal for both Isp values)")

    print("\n== D. Cold-atom equivalent temperature, rubidium-87 ==")
    for t_pk in (9.7, 38.0, 52.0):
        s = sigma_v(t_pk * 1e-12)
        print(f"   T = {t_pk:5.1f} pK: velocity spread {s*1e6:6.1f} um/s; spread after 7 s ~ {s*7.0*1e3:.3f} mm")
    print(f"   Velocity-spread ratio, 9.7 pK vs 38 pK: {sigma_v(9.7e-12)/sigma_v(38e-12):.3f}")
    print(f"   Check: T from sigma_v(9.7 pK) = {equivalent_temp(sigma_v(9.7e-12))*1e12:.2f} pK")

    print("\n== F. Rarefied-gas regimes (hard-sphere, nitrogen-like d = 3.7 A and T = 300 K ASSUMED; L = 1 m) ==")
    for p_pa in (1e5, 1.0, 0.1, 1e-2, 1e-4):
        kn = knudsen(p_pa, 1.0)
        print(f"   p = {p_pa:9.0e} Pa: mean free path {mean_free_path_m(p_pa):10.3e} m, Kn = {kn:10.3e}  -> {regime(kn)}")

    print("\n== G. Free fall and fleet screening ==")
    print(f"   Free fall in 1 s: {free_fall_m(1.0):.1f} m ; in 7 s: {free_fall_m(7.0):.0f} m")
    for n in (10, 100, 1000):
        print(f"   Fleet of {n:4d}: {screening_pairs(n):8,d} pairs to screen")

    print("\n== H. Arithmetic checks on published claims ==")
    print(f"   Flexcompute: 78 h / 0.4 GPU-h = {78.0/0.4:.0f}x ; 0.4 GPU-hours = {0.4*60:.0f} minutes")
    tw = (715.0 / 640.0) / 0.70
    print(f"   Northstar vs BE-4 (company figures): thrust x{715/640:.3f}, mass x0.70 -> thrust-to-weight x{tw:.2f}")

## 9. The self-test
Eighteen checks against textbook values and mathematical properties (for example, that the power-optimal divergence is a true maximum).

In [ ]:
def selftest():
    assert abs(fspl_db(1e3, 1e9) - 92.45) < 0.01                      # textbook constant
    assert abs(circ_speed(R_EARTH + 400.0) - 7.67) < 0.02              # LEO speed
    assert abs(circ_speed(R_GEO) - 3.075) < 0.01                       # GEO speed
    assert 3.8 < sum(hohmann_dv(R_EARTH + 185.0, R_GEO, 0.0)) < 4.1    # LEO->GEO planar
    assert 3.0 < tli_dv(R_EARTH + 185.0) < 3.3                         # TLI
    assert abs(slant_range_km(500.0, 90.0) - 500.0) < 1e-6             # zenith pass
    assert slant_range_km(500.0, 10.0) > slant_range_km(500.0, 30.0)
    assert abs(math.log(mass_ratio(3.0, 460.0)) * 460.0 * G0 / 1e3 - 3.0) < 1e-9
    assert abs(optimal_half_divergence(15e-6) - 21.2132e-6) < 1e-9
    th = optimal_half_divergence(15e-6)
    f = lambda t: (1.0 / t ** 2) * pointing_loss(15e-6, t)
    assert f(th) > f(0.8 * th) and f(th) > f(1.2 * th)                 # it is a maximum
    assert abs((35.0 / 15.0) ** 2 - 5.4444) < 1e-3
    assert abs(equivalent_temp(sigma_v(9.7e-12)) - 9.7e-12) < 1e-24
    assert rf_max_rate_bps(10e6, 35.0)["rate_bps"] > rf_max_rate_bps(10e6, 15.0)["rate_bps"]
    assert abs(rf_max_rate_bps(1e6, 15.0)["rate_bps"] / rf_max_rate_bps(2e6, 15.0)["rate_bps"] - 4.0) < 1e-6
    assert abs(mean_free_path_m(1e5) - 6.8e-8) < 1e-8                  # ~68 nm at 1 atm
    assert regime(knudsen(1e5, 1.0)) == "continuum" and regime(knudsen(1e-4, 1.0)) == "free-molecular"
    assert abs(free_fall_m(1.0) - 4.903325) < 1e-6
    assert screening_pairs(1000) == 499_500 and screening_pairs(10) == 45
    print("\nSELF-TEST PASSED (18 checks)")

## 10. Run it
Runs the report, then the self-test.

In [ ]:
report()
selftest()